In [1]:
!pip install langchain langchain-community langchain-text-splitters langchain-google-genai faiss-cpu pypdf sentence-transformers gradio

In [2]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"]=userdata.get("GOOGLE_API_KEY")
print("key Loaded");

key Loaded


In [3]:
from google.colab import files
uploaded=files.upload()
pdf_path=list(uploaded.keys())[0]
print("Uploaded:",pdf_path)

Saving lbdl.pdf to lbdl (1).pdf
Uploaded: lbdl (1).pdf


In [5]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader(pdf_path)
pages=loader.load()
print("Total Pages: ",len(pages))

Total Pages:  189


In [7]:
print(pages[2].page_content[:500])

Contents
Contents 5
List of figures 7
Foreword 8
I Foundations 10
1 Machine Learning 11
1.1 Learning from data . . . . . . . 12
1.2 Basis function regression . . . . 14
1.3 Under and overfitting . . . . . . 16
1.4 Categories of models . . . . . . 18
2 Efficient Computation 20
2.1 GPUs, TPUs, and batches . . . . 21
2.2 Tensors . . . . . . . . . . . . . . 23
3 Training 25
3.1 Losses . . . . . . . . . . . . . . 26
3.2 Autoregressive models . . . . . 30
3.3 Gradient descent . . . . . . . . 35
3


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(pages)
print("Total chunks:",len(chunks))
print(chunks[2].page_content[:300])


Total chunks: 314
Contents
Contents 5
List of figures 7
Foreword 8
I Foundations 10
1 Machine Learning 11
1.1 Learning from data . . . . . . . 12
1.2 Basis function regression . . . . 14
1.3 Under and overfitting . . . . . . 16
1.4 Categories of models . . . . . . 18
2 Efficient Computation 20
2.1 GPUs, TPUs, and bat


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db=FAISS.from_documents(chunks,embeddings)
retriever=db.as_retriever(search_kwargs={"k":4})
print("Vector database ready")

/tmp/ipykernel_9884/1212223936.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database ready


In [11]:
from google.colab import userdata
import os
from google import genai
os.environ["GOOGLE_API_KEY"]=userdata.get("GOOGLE_API_KEY")
client=genai.Client(api_key=os.environ["GOOGLE_API_KEY"])


In [30]:
def answer_question(question):
  docs=retriever.invoke(question)
  context="\n\n".join(f"[Page{d.metadata.get('page',0)+1}]\n{d.page_content}" for d in docs)
  prompt=f"""You are a helpful teaching assistant. Use the context below from a PDF to answer the question.
  Rules:
  -Give a detailed,well explained answers in several paragraphs or bullet points.
  -Explain the concept step by step,with the definition,how it works and an example if the context has one.
  -Answer only using the context below.If the answer is not in the context say "I could not find that in the PDF."

  Context:
  {context}
  Question:{question}"""
  response=client.models.generate_content(model="gemini-flash-lite-latest",contents=prompt)
  pages_used=sorted({d.metadata.get("page",0)+1 for d in docs})
  return response.text+f"\n\nSource pages:{pages_used}"

In [31]:
print(answer_question("What is Reinforcement learning"))

Based on the provided context, here is a detailed, step-by-step explanation of Reinforcement Learning:

* **Definition:** Reinforcement Learning (RL) is a standard setup used to solve problems—such as strategy games or robotic control—that can be formalized using a discrete-time state process ($S_t$) and a reward process ($R_t$) modulated by choosing actions ($A_t$). 

* **How it Works (The Framework and Objective):**
  * **Markov Decision Process (MDP):** If the state $S_t$ is Markovian—meaning it carries alone as much information about the future as all the past states until that instant—the object is called a Markov Decision Process (MDP).
  * **The Objective:** Given an MDP, the classical objective is to find a policy ($\pi$) such that the action $A_t = \pi(S_t)$ maximizes the expectation of the return. 
  * **The Return:** The return is defined as an accumulated discounted reward represented by the formula: 
    $$\mathbb{E} \left[ \sum_{t \ge 0} \gamma^t R_t \right]$$
    where $

In [ ]:
import gradio as gr
def chat(message,history):
  return answer_question(message)
gr.ChatInterface(fn=chat,
                 title="PDF Chatbot",
                 chatbot=gr.Chatbot(
                     latex_delimiters=[
                         {"left":"$$","right":"$$","display":True},
                         {"left":"$","right":"$","display":False},
                     ]
                 ),
                description="Ask any Question about your pdf."
                ).launch(share=True,debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://859746f68084f6f7d7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
for m in client.models.list():

    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l